# V2.1-P: Frozen package import (step 6)

This text-only validation notebook reads the aggregate freeze result from
`temp/v2/v2_package.json` and its public manifest at `config/v2_results.json`.
It contains no plots, dataframes, or modeling.

Step 6, governed by `docs/ADR-013-v2-frozen-package.md`, freezes the complete
V2 hierarchical package: critical detector, calibrated threshold, reference to
the frozen S7 package, and combination rule. Raw logs and transient Kaggle
directories are not part of the publication.


## Why the reproduction gate exists

No fitted model was persisted during V2.1-D1: the benchmark published
aggregates only and discarded its thirty estimators with the session.
Freezing the package therefore requires refitting the selected
candidate, and refitting creates the possibility that the frozen
artifact differs from the object D1 actually measured.

So the freeze is conditional on an exact reproduction gate, with no
numeric tolerance at all. Under D1's own code path, the refit must
reproduce the calibrated threshold (-0.13949530151425016), both
confusion matrices, the positive-decision and effective-override
counts of the two windows (57 and 16 on calibration, 258 and 82 on
the outer window), and the hard-negative pool counts (946 positives
and 14190 hard negatives). Every comparison is recorded as its own
named boolean.

The comparison is exact on the aggregate checks. It demonstrates behavioral reproduction on the recorded measures, but it does not prove row-level pool identity because D1 persisted no equivalent signature.

There are exactly two outcomes:

- `PACKAGE_FROZEN`: every check passed, and the joblib bundle was
  written to `artifacts/v2/consumer_complaint_detector_v2.joblib`;
- `REPRODUCTION_MISMATCH`: at least one check failed, NO bundle was
  written, and the divergence is published as evidence instead.

A mismatch is not an operational nuisance to be retried away. Nothing
is ever frozen silently over numbers that do not match, and the
decision on how to proceed returns to the Data Scientist. The report
below states which of the two outcomes happened before it prints
anything else.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from consumer_complaint_intelligence.v2_import import render_package_import_report

In [2]:
PACKAGE_RESULT_PATH = PROJECT_ROOT / 'temp' / 'v2' / 'v2_package.json'
PACKAGE_MANIFEST_PATH = PROJECT_ROOT / 'config' / 'v2_results.json'


In [3]:
print(render_package_import_report(
    PACKAGE_RESULT_PATH,
    manifest_path=PACKAGE_MANIFEST_PATH,
))


=== V2.1-P FROZEN PACKAGE RESULT ===
V2.1-P FROZEN HIERARCHICAL PACKAGE

OUTCOME
  outcome: PACKAGE_FROZEN
  frozen: True
  complete: True
  status: COMPLETE
  run_mode: full
  runtime_seconds: 1309.026400
  deployment:
    deployment_authorized: False
    status: FROZEN_FOR_CONFIRMATION
    next_step: open_stress_2025_h2_once_under_a_new_confirmatory_protocol

  PACKAGE FROZEN: every reproduction check passed and the
  fitted joblib bundle was written.

REPRODUCTION GATE (EXACT, NO TOLERANCE)
  passed: PASS
  check_count: 21
  comparison: exact_no_tolerance
  source_of_truth: temp/v2/v2_classical_benchmark.json
  candidate_id: word_char_tfidf_union_40000_60000_c_1_hard_negative
  checks:
     check                                            verdict
  -  -----------------------------------------------  -------
  *  calibrated_threshold                             PASS   
  *  calibration_confusion_matrix                     PASS   
  *  calibration_override_decisions                   

## Reading the report

The report is plain text and each `=== ... ===` section covers one
input. Inside the result section, OUTCOME comes first and states in
words whether a bundle was persisted. REPRODUCTION GATE lists every
check as PASS or FAIL, marking with `*` the canonical checks named in
`required_checks`, and prints a DIVERGENCES block whenever a check
failed.

A confusion-matrix divergence is shown only as an aggregate difference
summary: mismatched cells, total and maximum absolute difference, and
the two totals. The matrices themselves are never printed, and neither
are narratives, identifiers, or row indices.

SAFETY MARGIN repeats what the margin does not say. The headroom was
measured on the same outer window that served as the selection
surface, first among the eligible candidates and then in the D2
challenge, so it is development-optimistic and is not independent
evidence of future performance. The only independent evidence will
come from step 7, which this notebook neither authorizes nor performs.

INTEGRITY closes with the declared boundary. Note that
`persists_fitted_weights` is deliberately true here, because freezing
a package is precisely persisting fitted weights, while
`persists_narratives_or_identifiers` stays false.

This notebook decides nothing on its own; it only makes the published
evidence legible for human review.